# Tree of shapes and residual trees on `hydrant_2_3bit.png`

This compact tutorial uses `mmcfilters` to construct and filter three self-dual hierarchies, and `mtviz` to display the image, the complete trees, selected node supports, and filtered results.

The accompanying paper provides the definitions and proofs. The notebook stays close to the public APIs and intentionally avoids reimplementing tree algorithms in Python.

## Goal

The notebook has four steps:

1. load `dat/hydrant_2_3bit.png`;
2. build a tree of shapes and two residual trees;
3. display and inspect the complete trees with `mtviz`; and
4. interactively select an attribute and threshold, then compare the filtered images.

## Setup

Run from the repository root or from `notebooks/`, using the project environment described in `README.md` and `notebooks/requirements.txt`. Pixel coordinates use `(row, column)` order.

In [1]:
from pathlib import Path

import cv2
import ipywidgets as widgets
import mmcfilters
import mtviz as viz
import numpy as np
import pandas as pd
from IPython.display import clear_output, display
from bokeh.io import output_notebook
from bokeh.layouts import row as bokeh_row

output_notebook(hide_banner=True)

IMAGE_PATH = Path("dat/hydrant_2_3bit.png")
if not IMAGE_PATH.exists():
    IMAGE_PATH = Path("../dat/hydrant_2_3bit.png")

image = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_GRAYSCALE)
assert image is not None, f"Could not read {IMAGE_PATH}"
image = np.ascontiguousarray(image, dtype=np.uint8)
rows, columns = image.shape

display(
    pd.Series(
        {
            "path": IMAGE_PATH.as_posix(),
            "shape": f"{rows} x {columns}",
            "pixels": image.size,
            "gray levels": np.unique(image).size,
        },
        name="hydrant_2_3bit.png",
    ).to_frame()
)

,hydrant_2_3bit.png
path,../dat/hydrant_2_3bit.png
shape,352 x 363
pixels,127776
gray levels,5


## 1. Build the hierarchies

| Hierarchy | Construction used here |
| --- | --- |
| Tree of Shapes | default complementary Min4/Max8 convention |
| Saturated Residual | saturated eligible extrema, shared 8-adjacency, exterior pixel 0 |
| Unrestricted Residual | all eligible regional extrema, shared 8-adjacency |

The paper's tree-of-shapes equivalence concerns a complementary-adjacency specialization. The shared-adjacency saturated residual tree below is therefore compared with the tree of shapes, not assumed to be identical to it.

In [2]:
RESIDUAL_RADIUS = 1.5      # shared 8-adjacency
INFINITY_PIXEL = 0         # upper-left pixel in row-major order

factory = mmcfilters.MorphologicalTreeFactory

trees = {
    "Tree of Shapes": factory.create_tree_of_shapes(
        image,
        infinity_pixel=INFINITY_PIXEL,
    ),
    "Saturated Residual": factory.create_saturated_residual_tree(
        image,
        infinity_pixel=INFINITY_PIXEL,
        radius=RESIDUAL_RADIUS,
    ),
    "Unrestricted Residual": factory.create_unrestricted_residual_tree(
        image,
        radius=RESIDUAL_RADIUS,
    ),
}


display(
    pd.DataFrame(
        [
            {
                "hierarchy": name,
                "nodes": tree.num_nodes,
                "leaves": len(tree.leaves),
                "root altitude": int(tree.node_altitude(tree.root)),
            }
            for name, tree in trees.items()
        ]
    ).set_index("hierarchy")
)

,nodes,leaves,root altitude
hierarchy,,,
Tree of Shapes,81,36,219
Saturated Residual,81,36,219
Unrestricted Residual,80,36,109


## 2. Select a pixel and inspect its nodes

`mtviz.makePlotImageTreeInspector` turns the input image into a pixel selector. Click any pixel: the selected `(row, column)`, linear index, gray level, node attributes, and the three node supports update immediately in the browser.

The interaction uses the `selection_source` supplied by `mtviz`; it does not require a fixed `INSPECTION_POINT`, rerunning the cell, or a Bokeh server.

In [3]:
INPUT_VIEW_WIDTH = 520
SUPPORT_VIEW_WIDTH = 390
SUPPORT_ALPHA = 120

viz.show(
    viz.makePlotImageTreeInspector(
        image,
        trees,
        title="Click a pixel in hydrant_2_3bit.png",
        width=INPUT_VIEW_WIDTH,
        support_width=SUPPORT_VIEW_WIDTH,
        alpha=SUPPORT_ALPHA,
    )
)

## 3. Display the complete trees with `mtviz`

The next cell displays **all three trees**. Each node is labeled by its local node ID.

- Hover over a node to read its altitude, signed residue, support area, proper-part area, and number of children.
- Click a node to highlight it.
- Use the node-size and view-scale sliders, pan, and box zoom to inspect dense branches.

Node IDs are local to a hierarchy and should not be compared across trees.

In [4]:
TREE_NODE_SIZE = 14
TREE_PANEL_WIDTH = 1050

areas = {
    name: mmcfilters.Attribute.compute_single_topology_attribute(
        tree,
        mmcfilters.Attribute.AREA,
        dtype=np.float64,
    )
    for name, tree in trees.items()
}

for tree_name, tree in trees.items():
    area = areas[tree_name]
    viz.show(
        viz.makePlotTree(
            tree.root,
            tree.children,
            {
                "altitude": tree.node_altitude,
                "residue": tree.node_residue,
                "support area": lambda node, area=area: int(area[node]),
                "proper-part area": tree.proper_part_cardinality,
                "children": tree.num_children,
            },
            int,
            node_size=TREE_NODE_SIZE,
            panel_width=TREE_PANEL_WIDTH,
            panel_caption=f"{tree_name} - {tree.num_nodes} nodes",
        )
    )

## 4. Filter the trees and display the images

Choose one of the six attributes in the combo box and its pruning-min threshold with the slider. The slider range adapts to the selected attribute; `mmcfilters.Attribute` computes the node values, `mmcfilters.DirectAttributeFilter` performs the filtering, and `mtviz` displays the three reconstructed images. Set the threshold to zero to recover the unfiltered image.

In [5]:
FILTER_VIEW_WIDTH = 390

ATTRIBUTE_CONTROLS = {
    "AREA": {
        "attribute": mmcfilters.Attribute.AREA,
        "step": 1.0,
        "initial": float(np.ceil(0.005 * image.size)),
        "format": ".0f",
    },
    "CIRCULARITY": {
        "attribute": mmcfilters.Attribute.CIRCULARITY,
        "step": 0.01,
        "initial": 0.50,
        "format": ".2f",
    },
    "INERTIA": {
        "attribute": mmcfilters.Attribute.INERTIA,
        "step": 0.05,
        "initial": 0.50,
        "format": ".2f",
    },
    "RECTANGULARITY": {
        "attribute": mmcfilters.Attribute.RECTANGULARITY,
        "step": 0.01,
        "initial": 0.50,
        "format": ".2f",
    },
    "BOUNDING_BOX_HEIGHT": {
        "attribute": mmcfilters.Attribute.BOUNDING_BOX_HEIGHT,
        "step": 1.0,
        "initial": 10.0,
        "format": ".0f",
    },
    "MAX_DIST": {
        "attribute": mmcfilters.Attribute.MAX_DIST,
        "step": 1.0,
        "initial": 5.0,
        "format": ".0f",
    },
}

attribute_values = {
    attribute_name: {
        tree_name: mmcfilters.Attribute.compute_single_topology_attribute(
            tree, specification["attribute"], dtype=np.float64
        )
        for tree_name, tree in trees.items()
    }
    for attribute_name, specification in ATTRIBUTE_CONTROLS.items()
}

for attribute_name, specification in ATTRIBUTE_CONTROLS.items():
    specification["maximum"] = max(
        float(np.max(values))
        for values in attribute_values[attribute_name].values()
    )

attribute_selector = widgets.Dropdown(
    options=list(ATTRIBUTE_CONTROLS),
    value="AREA",
    description="Attribute:",
)
threshold_slider = widgets.FloatSlider(
    description="Threshold:",
    continuous_update=False,
    readout=True,
    layout=widgets.Layout(width="600px"),
)
filter_output = widgets.Output()
updating_controls = False

def configure_threshold(attribute_name):
    global updating_controls
    updating_controls = True
    specification = ATTRIBUTE_CONTROLS[attribute_name]
    try:
        threshold_slider.min = 0.0
        threshold_slider.max = specification["maximum"]
        threshold_slider.step = specification["step"]
        threshold_slider.readout_format = specification["format"]
        threshold_slider.value = min(
            specification["initial"], specification["maximum"]
        )
    finally:
        updating_controls = False

def render_filtered_images(*_):
    if updating_controls:
        return

    attribute_name = attribute_selector.value
    if attribute_name not in ATTRIBUTE_CONTROLS:
        return

    threshold = threshold_slider.value
    panels = []
    summary_rows = []

    for tree_name, tree in trees.items():
        values = attribute_values[attribute_name][tree_name]
        keep = (values > threshold).tolist()
        keep[tree.root] = True
        preservation_mask = mmcfilters.NodePreservationMask(keep)
        filtered_image = mmcfilters.DirectAttributeFilter(tree).apply(preservation_mask)
        
        panels.append(
            viz.makePlotImage(
                filtered_image,
                title=f"{tree_name} | {attribute_name} >= {threshold:g}",
                width=FILTER_VIEW_WIDTH,
            ).panel
        )
        summary_rows.append(
            {
                "hierarchy": tree_name,
                "nodes meeting threshold": int(np.count_nonzero(values >= threshold)),
                "changed pixels": int(np.count_nonzero(filtered_image != image)),
                "remaining gray levels": int(np.unique(filtered_image).size),
            }
        )

    with filter_output:
        clear_output(wait=True)
        viz.show(bokeh_row(*panels))
        display(pd.DataFrame(summary_rows).set_index("hierarchy"))

def select_attribute(change):
    attribute_name = change["new"]
    if attribute_name in ATTRIBUTE_CONTROLS:
        configure_threshold(attribute_name)
        render_filtered_images()

attribute_selector.observe(select_attribute, names="value")
threshold_slider.observe(render_filtered_images, names="value")

configure_threshold(attribute_selector.value)
display(widgets.HBox([attribute_selector, threshold_slider]), filter_output)
render_filtered_images()

Output()